# 2. Feature selection

Runs the standard GA and the five modified configurations on the features from notebook 1,
then compares the standard GA with filter rankings and random subsets of the same size.
GA fitness is the inner 3-fold CV macro-F1 of the weighted ensemble on outer-training images only.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
from joblib import Parallel, delayed
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

DATASET = "BUSI"  # or "BUS-UCLM"
OUT_DIR = Path("../outputs") / DATASET

SEED = 42
POP, MAX_GEN, PATIENCE, INNER = 30, 50, 5, 3
WEIGHTS = [1, 1, 2, 1]
CONFIGS = ["Standard", "Naskar_ab", "Naskar_c", "OBL_init", "OBL_jump", "OBL_both"]
GA_SEEDS = [0, 1] if DATASET == "BUSI" else [0]
COND_ID = {"raw": 0, "predicted": 1, "oracle": 2, "shuffled": 9}

data = joblib.load(OUT_DIR / "folds.pkl")
y, groups, folds = data["y"], data["groups"], data["folds"]


def features(condition, k):
    return np.load(OUT_DIR / "features" / f"{condition}_fold{k}.npz")["X"]

## Ensemble

In [ ]:
def classifiers():
    return [DecisionTreeClassifier(random_state=0),
            KNeighborsClassifier(5),
            RandomForestClassifier(100, random_state=0, n_jobs=1),
            HistGradientBoostingClassifier(max_iter=100, max_depth=3, max_bins=64,
                                           early_stopping=False, random_state=0)]


def probas(X_train, y_train, X_test):
    return np.stack([c.fit(X_train, y_train).predict_proba(X_test) for c in classifiers()])


def vote(prob, weights):
    return np.tensordot(np.asarray(weights) / np.sum(weights), prob, 1).argmax(-1)


def cv_f1(X, y_, inner):
    prob = np.zeros((4, len(y_), 3))
    for a, b in inner:
        prob[:, b] = probas(X[a], y_[a], X[b])
    pred = vote(prob, WEIGHTS)
    return np.mean([f1_score(y_[b], pred[b], average="macro") for _, b in inner])

## GA operators

Naskar (a) arithmetic crossover, (b) entropy-scaled roulette selection, (c) k-means initialisation;
OBL adds opposite chromosomes at initialisation, during the run (generation jumping), or both.

In [ ]:
def init_random(n, rng):
    pop = []
    for _ in range(POP):
        c = np.zeros(n, np.int8)
        c[rng.choice(n, rng.integers(int(0.3 * n), int(0.6 * n) + 1), replace=False)] = 1
        pop.append(c)
    return pop


def init_kmeans(X, rng):
    sim = cosine_similarity(X.T)
    ks = range(2, 21)
    inertia = np.array([KMeans(k, random_state=42, n_init=10).fit(sim).inertia_ for k in ks])
    xs = np.linspace(0, 1, len(inertia))
    ys = (inertia - inertia.min()) / (np.ptp(inertia) + 1e-12)
    k = list(ks)[np.argmax(np.abs(xs + ys - 1))]
    labels = KMeans(k, random_state=42, n_init=10).fit_predict(sim)
    pop = []
    for _ in range(POP):
        c = np.zeros(X.shape[1], np.int8)
        for g in range(k):
            members = np.flatnonzero(labels == g)
            c[rng.choice(members, rng.integers(1, max(1, len(members) // 2) + 1), replace=False)] = 1
        pop.append(c)
    return pop


def roulette(pop, fit, rng, adaptive=False):
    f = np.array(fit)
    if adaptive:
        _, counts = np.unique(f.round(6), return_counts=True)
        q = counts / counts.sum()
        diversity = -(q * np.log2(q)).sum() / np.log2(len(f))
        f = f / (1 + np.exp(-10 * (diversity - 0.5))) + 0.1 / len(f)
    return [pop[i].copy() for i in rng.choice(len(pop), len(pop), p=f / f.sum())]


def two_point(a, b, rng):
    i, j = sorted(rng.choice(len(a) + 1, 2, replace=False))
    return np.r_[a[:i], b[i:j], a[j:]], np.r_[b[:i], a[i:j], b[j:]]


def arithmetic(a, b, rng, s=10000):
    n = len(a)
    A, B = int("".join(map(str, a)), 2), int("".join(map(str, b)), 2)
    children = []
    for p, q in ((A, B), (B, A)):
        lam, eps = int(rng.integers(0, s + 1)), int(rng.integers(-s, s + 1))
        v = min(max((lam * p + (s - lam) * q + eps * (p - q)) // s, 0), (1 << n) - 1)
        children.append(np.array(list(format(v, f"0{n}b")), np.int8))
    return children


def mutate(c, rng):
    c = c.copy()
    flip = rng.random(len(c)) < 1 / len(c)
    c[flip] = 1 - c[flip]
    if c.sum() == 0:
        c[rng.integers(len(c))] = 1
    return c


def survivors(pop, fit):
    order = np.argsort(-np.array(fit), kind="stable")[:POP]
    return [pop[i] for i in order], [fit[i] for i in order]

In [ ]:
def genetic_search(X, y_, g, config, rng, k):
    inner = list(StratifiedGroupKFold(INNER, shuffle=True, random_state=k).split(y_, y_, g))
    cache = {}

    def fitness(c):
        key = np.packbits(c).tobytes()
        if key not in cache:
            cache[key] = cv_f1(X[:, np.flatnonzero(c)], y_, inner)
        return cache[key]

    adaptive = arith = config == "Naskar_ab"
    pop = init_kmeans(X, rng) if config == "Naskar_c" else init_random(X.shape[1], rng)
    if config in ("OBL_init", "OBL_both"):
        pop += [1 - c for c in pop]
    pop, fit = survivors(pop, [fitness(c) for c in pop])
    best, best_fit, last_improvement, stall = pop[0], fit[0], 0, 0

    for gen in range(1, MAX_GEN + 1):
        parents = roulette(pop, fit, rng, adaptive)
        crossover = arithmetic if arith else two_point
        children = [mutate(c, rng) for i in range(0, POP - 1, 2) for c in crossover(parents[i], parents[i + 1], rng)]
        pop, fit = survivors(pop + children, fit + [fitness(c) for c in children])
        if config in ("OBL_jump", "OBL_both") and rng.random() < 0.3:
            opposite = [1 - c for c in pop]
            pop, fit = survivors(pop + opposite, fit + [fitness(c) for c in opposite])
        if fit[0] > best_fit:
            best, best_fit, last_improvement, stall = pop[0], fit[0], gen, 0
        else:
            stall += 1
            if stall >= PATIENCE:
                break
    return best, last_improvement

## GA runs

Each run selects a subset on the outer-training images and classifies the outer-test fold once.
The full set (6 configurations, 3 conditions, 5 folds, plus the shuffled control) takes many CPU hours.

In [ ]:
def run(condition, k, config, seed):
    tr, te = folds[k]
    X = features(condition, k)
    rng = np.random.default_rng([SEED, k, seed, COND_ID[condition]])
    mask, last_improvement = genetic_search(X[tr], y[tr], groups[tr], config, rng, k)
    idx = np.flatnonzero(mask)
    pred = vote(probas(X[tr][:, idx], y[tr], X[te][:, idx]), WEIGHTS)
    return dict(condition=condition, fold=k, config=config, seed=seed,
                mask=mask, last_improvement=last_improvement, pred=pred)


jobs = [(c, k, g, s) for s in GA_SEEDS for g in CONFIGS for k in range(len(folds)) for c in ["raw", "predicted", "oracle"]]
jobs += [("shuffled", k, "Standard", 0) for k in range(len(folds))]
runs = Parallel(n_jobs=-1)(delayed(run)(*job) for job in jobs)
joblib.dump(runs, OUT_DIR / "ga_runs.pkl")

## Baselines

Filter rankings and random subsets use the subset size *k* of the standard GA (seed 0) in the same fold.

In [ ]:
def evaluate(X, tr, te, idx):
    pred = vote(probas(X[tr][:, idx], y[tr], X[te][:, idx]), WEIGHTS)
    return f1_score(y[te], pred, average="macro")


def baselines(condition, k):
    tr, te = folds[k]
    X = features(condition, k)
    ga = next(r["mask"] for r in runs if (r["condition"], r["fold"], r["config"], r["seed"]) == (condition, k, "Standard", 0))
    size = int(ga.sum())
    anova = np.nan_to_num(f_classif(X[tr], y[tr])[0])
    mi = mutual_info_classif(X[tr], y[tr], random_state=0)
    rng = np.random.default_rng(1000 + k)
    random_f1 = [evaluate(X, tr, te, np.sort(rng.choice(X.shape[1], size, replace=False))) for _ in range(10)]
    return dict(condition=condition, fold=k, k=size,
                all_512=evaluate(X, tr, te, np.arange(X.shape[1])),
                standard_ga=evaluate(X, tr, te, np.flatnonzero(ga)),
                anova=evaluate(X, tr, te, np.argsort(-anova)[:size]),
                mutual_info=evaluate(X, tr, te, np.argsort(-mi)[:size]),
                random=np.mean(random_f1))


rows = Parallel(n_jobs=-1)(delayed(baselines)(c, k) for c in ["raw", "predicted", "shuffled", "oracle"] for k in range(len(folds)))
joblib.dump(rows, OUT_DIR / "baselines.pkl")